In [ ]:
!pip install -q transformers datasets accelerate evaluate sentencepiece matplotlib seaborn scikit-learn

import transformers
print(transformers.__version__)

import random, numpy as np, torch, time
from datasets import load_dataset, Value
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)
import evaluate
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, precision_recall_curve, auc

In [ ]:
# ==================== GOOGLE DRIVE & LOAD SHARED DATA ====================
from google.colab import drive
import pandas as pd
from datasets import Dataset

drive.mount("/content/drive")

SHARED_DIR = "/content/drive/MyDrive/Colab Notebooks/IE4483 materials/processed_data"

train = Dataset.from_pandas(pd.read_json(f"{SHARED_DIR}/train_split.json"))
val = Dataset.from_pandas(pd.read_json(f"{SHARED_DIR}/val_split.json"))
test = Dataset.from_pandas(pd.read_json(f"{SHARED_DIR}/test_split.json"))

print(f"Training: {len(train)}")
print(f"Validation: {len(val)}")
print(f"Test: {len(test)}")

In [ ]:
# ==================== SETUP ====================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU Only")
assert torch.cuda.is_available(), "GPU not enabled!"

In [ ]:
# ==================== HELPER FUNCTIONS 1====================
def prepare_dataset(train, val, test, tokenizer):
    """Tokenize and prepare datasets for Sentiment Classification"""
    def preprocess(batch):
        # Tokenize 'reviews' text
        return tokenizer(batch["reviews"], truncation=True, max_length=128)

    def add_labels_train(example):
        example["labels"] = int(example["sentiments"])
        return example

    # Tokenize the datasets
    train_enc = train.map(preprocess, batched=True, remove_columns=["reviews"])
    val_enc = val.map(preprocess, batched=True, remove_columns=["reviews"])
    test_enc = test.map(preprocess, batched=True, remove_columns=["reviews"])

    # Set 'labels'
    train_enc = train_enc.map(add_labels_train, remove_columns=["sentiments"])
    val_enc = val_enc.map(add_labels_train, remove_columns=["sentiments"])

    # Set format to PyTorch tensors
    train_enc.set_format("torch", columns=['input_ids', 'attention_mask', 'labels'])
    val_enc.set_format("torch", columns=['input_ids', 'attention_mask', 'labels'])
    test_enc.set_format("torch", columns=["input_ids", "attention_mask"])

    return train_enc, val_enc, test_enc

In [ ]:
# ==================== HELPER FUNCTIONS 2====================
def get_compute_metrics():
    accuracy = evaluate.load("accuracy")
    precision = evaluate.load("precision")
    recall = evaluate.load("recall")
    f1 = evaluate.load("f1")

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = logits.argmax(-1)
        return {
            "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
            "precision": precision.compute(predictions=preds, references=labels, average="binary")["precision"],
            "recall": recall.compute(predictions=preds, references=labels, average="binary")["recall"],
            "f1": f1.compute(predictions=preds, references=labels, average="binary")["f1"],
        }
    return compute_metrics

In [ ]:
# ==================== ANALYSIS FUNCTION 1====================
def plot_training_curves(trainer, model_name):
    logs = trainer.state.log_history
    train_loss = [log['loss'] for log in logs if 'loss' in log]
    eval_loss = [log['eval_loss'] for log in logs if 'eval_loss' in log]
    eval_f1 = [log['eval_f1'] for log in logs if 'eval_f1' in log]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(train_loss, 'b-o', label='Train Loss')
    if len(eval_loss) > 0:
        epochs = np.linspace(0, len(train_loss)-1, len(eval_loss))
        ax1.plot(epochs, eval_loss, 'r-s', label='Eval Loss')
    ax1.set_xlabel('Steps/Epochs')
    ax1.set_ylabel('Loss')
    ax1.set_title(f'{model_name} - Loss')
    ax1.legend()
    ax1.grid(alpha=0.3)

    ax2.plot(range(1, len(eval_f1)+1), eval_f1, 'g-o')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('F1 Score')
    ax2.set_title(f'{model_name} - F1')
    ax2.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'/content/{model_name}_curves.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# ==================== ANALYSIS FUNCTION 2====================
def plot_confusion_matrix(model, tokenizer, test_enc, model_name):
    device = model.device
    all_preds, all_labels = [], []

    for example in test_enc:
        inputs = {k: v.unsqueeze(0).to(device) for k, v in example.items() if k != 'labels'}
        with torch.no_grad():
            logits = model(**inputs).logits
        all_preds.append(logits.argmax(-1).item())
        all_labels.append(example['labels'].item())

    cm = confusion_matrix(all_labels, all_preds)

    plt.figure(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Negative', 'Positive'],
                yticklabels=['Negative', 'Positive'])
    plt.ylabel('True')
    plt.xlabel('Predicted')
    plt.title(f'{model_name} - Confusion Matrix')
    plt.tight_layout()
    plt.savefig(f'/content/{model_name}_cm.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"TN (True Negative):{cm[0,0]} FP (False Positive):{cm[0,1]} | FN (False Negative):{cm[1,0]} TP (True Positive):{cm[1,1]}")

In [ ]:
# ==================== ANALYSIS FUNCTION 3====================
def plot_pr_curve(model, tokenizer, test_enc, model_name):
    device = model.device
    all_probs, all_labels = [], []

    for example in test_enc:
        inputs = {k: v.unsqueeze(0).to(device) for k, v in example.items() if k != 'labels'}
        with torch.no_grad():
            logits = model(**inputs).logits
        prob = torch.softmax(logits, dim=-1)[0, 1].item()
        all_probs.append(prob)
        all_labels.append(example['labels'].item())

    precision, recall, _ = precision_recall_curve(all_labels, all_probs)
    pr_auc = auc(recall, precision)

    plt.figure(figsize=(7, 5))
    plt.plot(recall, precision, 'b-', linewidth=2, label=f'AUC={pr_auc:.3f}')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title(f'{model_name} - PR Curve')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'/content/{model_name}_pr.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"PR-AUC: {pr_auc:.4f}")

In [ ]:
# ==================== ANALYSIS FUNCTION 4====================
def analyze_errors(model, tokenizer, test_data, model_name, n=5):
    """Simple error analysis"""
    device = model.device
    errors = []

    for i, example in enumerate(test_data):
        if len(errors) >= n * 2:
            break
        # Use 'reviews' and 'sentiments' key instead of text/toxicity
        if 'reviews' not in example or 'sentiments' not in example:
            continue
        text = example['reviews']
        true_label = int(example['sentiments'])

        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            logits = model(**inputs).logits
        pred_label = logits.argmax(-1).item()

        if pred_label != true_label:
            errors.append({
                'text': text[:80],
                'true': true_label,
                'pred': pred_label
            })

    print(f"\n{model_name} - Error Examples (showing {min(n, len(errors))}):")
    for i, e in enumerate(errors[:n], 1):
        print(f"{i}. '{e['text']}...'")
        print(f"   True: {'positive' if e['true']==1 else 'negative'} | Pred: {'positive' if e['pred']==1 else 'negative'}\n")

In [ ]:

# ==================== TRAIN & EVALUATE FUNCTION ====================
import os, json, time, shutil
import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments, Trainer,
    DataCollatorWithPadding, EarlyStoppingCallback
)
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report
)

def train_and_evaluate(
    checkpoint, model_name, train_enc, val_enc,
    test_enc, test_raw, tokenizer, config
):

    output_dir = (
    "/content/drive/MyDrive/Colab Notebooks/"
    f"IE4483_BERT/experiments/{model_name}"
    )
    os.makedirs(output_dir, exist_ok=True)

    set_seed(config["seed"])

    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint,
        num_labels=2,
        hidden_dropout_prob=config["dropout"],
        attention_probs_dropout_prob=config["dropout"],
        classifier_dropout=config["dropout"]
    )
    model.config.problem_type = "single_label_classification"

    # Freeze the first N encoder layers if requested
    frozen = config["frozen_layers"]
    for layer in model.bert.encoder.layer[:frozen]:
        for parameter in layer.parameters():
            parameter.requires_grad = False

    args = TrainingArguments(
        output_dir=output_dir,
        learning_rate=config["lr"],
        num_train_epochs=config["epochs"],
        per_device_train_batch_size=config["batch"],
        per_device_eval_batch_size=32,
        weight_decay=config["weight_decay"],
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        save_total_limit=2,
        report_to="none",
        logging_steps=50,
        seed=config["seed"],
        data_seed=config["seed"]
    )

    callbacks = []
    if config["early_stopping"]:
        callbacks.append(EarlyStoppingCallback(
            early_stopping_patience=2
        ))

    # Separate optimizer configuration
    optimizers = (None, None)
    if config["optimizer"] == "SGD":
        sgd = torch.optim.SGD(
            [p for p in model.parameters() if p.requires_grad],
            lr=config["lr"],
            momentum=0.9,
            weight_decay=config["weight_decay"]
        )
        optimizers = (sgd, None)

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_enc,
        eval_dataset=val_enc,
        processing_class=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=get_compute_metrics(),
        callbacks=callbacks,
        optimizers=optimizers
    )

    print(f"\n{'='*55}")
    print(f"TRAINING: {model_name}")
    print(config)
    print(f"{'='*55}")

    start = time.time()
    trainer.train()
    elapsed = (time.time() - start) / 60

    # Save selected model and tokenizer
    best_dir = os.path.join(output_dir, "best_model")
    trainer.save_model(best_dir)
    tokenizer.save_pretrained(best_dir)

    # Compute training and validation metrics
    train_out = trainer.predict(train_enc)
    val_out = trainer.predict(val_enc)
    test_out = trainer.predict(test_enc)

    train_pred = train_out.predictions.argmax(axis=-1)
    val_pred = val_out.predictions.argmax(axis=-1)
    test_pred = test_out.predictions.argmax(axis=-1)

    train_true = np.asarray(train_enc["labels"])
    val_true = np.asarray(val_enc["labels"])

    val_precision, val_recall, val_f1, _ = (
        precision_recall_fscore_support(
            val_true, val_pred,
            average="binary", zero_division=0
        )
    )

    neg_precision, neg_recall, neg_f1, _ = (
        precision_recall_fscore_support(
            val_true, val_pred,
            labels=[0], average="macro",
            zero_division=0
        )
    )

    metrics = {
        "Model": model_name,
        "LR": config["lr"],
        "Epochs": config["epochs"],
        "Batch": config["batch"],
        "Seed": config["seed"],
        "Optimizer": config["optimizer"],
        "Dropout": config["dropout"],
        "Weight Decay": config["weight_decay"],
        "Early Stopping": config["early_stopping"],
        "Frozen Layers": config["frozen_layers"],
        "Train Accuracy": accuracy_score(train_true, train_pred),
        "Validation Accuracy": accuracy_score(val_true, val_pred),
        "Precision": val_precision,
        "Recall": val_recall,
        "F1": val_f1,
        "Negative F1": neg_f1,
        "Negative Recall": neg_recall,
        "Best Epoch": trainer.state.best_model_checkpoint,
        "Time (min)": round(elapsed, 2)
    }

    # Save metrics and configuration
    with open(f"{output_dir}/metrics.json", "w") as f:
        json.dump(metrics, f, indent=2, default=str)

    with open(f"{output_dir}/config.json", "w") as f:
        json.dump(config, f, indent=2)

    pd.DataFrame(trainer.state.log_history).to_csv(
        f"{output_dir}/training_history.csv", index=False
    )

    # Save test predictions
    submission = pd.DataFrame({
        "reviews": list(test_raw["reviews"]),
        "predictions": test_pred.astype(int)
    })
    submission.to_csv(
        f"{output_dir}/submission.csv", index=False
    )

    # Save validation examples for error analysis
    val_df = pd.DataFrame({
        "reviews": list(val["reviews"]),
        "true_label": val_true,
        "prediction": val_pred
    })
    val_df["correct"] = (
        val_df["true_label"] == val_df["prediction"]
    )
    val_df.to_csv(
        f"{output_dir}/validation_predictions.csv",
        index=False
    )

    # Reuse existing analysis functions
    plot_training_curves(trainer, model_name)
    plot_confusion_matrix(
        trainer.model, tokenizer, val_enc, model_name
    )
    plot_pr_curve(
        trainer.model, tokenizer, val_enc, model_name
    )

    # Copy figures from Colab to Google Drive
    figures_dir = f"{output_dir}/figures"
    os.makedirs(figures_dir, exist_ok=True)

    for suffix in ["curves", "cm", "pr"]:
        filename = f"{model_name}_{suffix}.png"
        source = f"/content/{filename}"
        if os.path.exists(source):
            shutil.copy2(source, f"{figures_dir}/{filename}")

    print(f"\nCOMPLETED: {model_name}")
    print(f"Validation F1: {val_f1:.4f}")
    print(f"Negative F1: {neg_f1:.4f}")
    print(f"Saved to: {output_dir}")

    return metrics


In [ ]:

# ==================== MULTIPLE EXPERIMENTS ====================
import os, gc, json
import pandas as pd
import torch

checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

train_enc, val_enc, test_enc = prepare_dataset(
    train, val, test, tokenizer
)
base_dir = (
    "/content/drive/MyDrive/Colab Notebooks/"
    "IE4483_BERT/experiments"
)
os.makedirs(base_dir, exist_ok=True)

# Baseline configuration
baseline = {
    "lr": 3e-5,
    "epochs": 3,
    "batch": 16,
    "seed": 42,
    "optimizer": "AdamW",
    "dropout": 0.1,
    "weight_decay": 0.01,
    "early_stopping": False,
    "frozen_layers": 0
}

# Each run changes one main variable
experiments = [
    ("Run_01_Baseline", {}),
    ("Run_02_LR_2e5", {"lr": 2e-5}),
    ("Run_03_LR_5e5", {"lr": 5e-5}),
    ("Run_04_Epoch_4", {"epochs": 4}),
    ("Run_05_Seed_123", {"seed": 123}),

    ("Run_06_Batch_32", {
        "lr": 2e-5, "batch": 32
    }),

    ("Run_07_SGD", {
        "lr": 1e-3, "optimizer": "SGD"
    }),

    ("Run_08_Dropout", {
        "lr": 2e-5, "dropout": 0.3
    }),

    ("Run_09_WeightDecay", {
        "lr": 2e-5, "weight_decay": 0.05
    }),

    ("Run_10_EarlyStopping", {
        "lr": 2e-5,
        "epochs": 8,
        "early_stopping": True
    }),

    ("Run_11_FrozenLayers", {
        "lr": 2e-5, "frozen_layers": 6
    })
]

comparison_path = f"{base_dir}/comparison.csv"

for name, changes in experiments:

    config = {**baseline, **changes}
    run_dir = f"{base_dir}/{name}"

    metrics_path = f"{run_dir}/metrics.json"
    model_path = f"{run_dir}/best_model/model.safetensors"
    submission_path = f"{run_dir}/submission.csv"

    # Skip fully completed experiments
    if (
        os.path.exists(metrics_path)
        and os.path.exists(model_path)
        and os.path.exists(submission_path)
    ):
        print(f"SKIPPING COMPLETED: {name}")
        continue

    result = train_and_evaluate(
        checkpoint, name,
        train_enc, val_enc, test_enc,
        test, tokenizer, config
    )

    # Rebuild comparison after every completed run
    records = []
    for folder, _, files in os.walk(base_dir):
        if "metrics.json" in files:
            with open(os.path.join(folder, "metrics.json")) as f:
                records.append(json.load(f))

    if records:
        df = pd.DataFrame(records).sort_values("Model")
        df.to_csv(comparison_path, index=False)

    gc.collect()
    torch.cuda.empty_cache()

print("\nALL 11 EXPERIMENTS COMPLETED!")


In [ ]:

# ==================== FINAL COMPARISON ====================
import os, json, shutil
import pandas as pd

base_dir = (
    "/content/drive/MyDrive/Colab Notebooks/"
    "IE4483_BERT/experiments"
)

records = []

for name in sorted(os.listdir(base_dir)):
    metrics_path = f"{base_dir}/{name}/metrics.json"

    if os.path.isfile(metrics_path):
        with open(metrics_path) as f:
            records.append(json.load(f))

if not records:
    print("No completed experiments found.")
else:
    df = pd.DataFrame(records).sort_values("Model")
    df.to_csv(f"{base_dir}/comparison.csv", index=False)

    print("\n========== EXPERIMENT COMPARISON ==========")
    display(df)

    best = df.loc[df["F1"].idxmax()]
    best_name = best["Model"]

    print("\nBEST EXPERIMENT:", best_name)
    print("Validation Accuracy:", best["Validation Accuracy"])
    print("Validation F1:", best["F1"])
    print("Negative F1:", best["Negative F1"])

    # Check best submission
    submission_path = f"{base_dir}/{best_name}/submission.csv"
    submission = pd.read_csv(submission_path)

    print("\nSubmission rows:", len(submission))
    display(submission.head())

    # Prepare small files for GitHub
    export_dir = (
    "/content/drive/MyDrive/Colab Notebooks/"
    "IE4483_BERT/github_export"
    )
    os.makedirs(f"{export_dir}/results", exist_ok=True)
    os.makedirs(f"{export_dir}/figures", exist_ok=True)

    shutil.copy2(
        f"{base_dir}/comparison.csv",
        f"{export_dir}/results/comparison.csv"
    )

    for name in df["Model"]:
        run_dir = f"{base_dir}/{name}"

        for filename in [
            "metrics.json",
            "training_history.csv",
            "validation_predictions.csv"
        ]:
            src = f"{run_dir}/{filename}"
            if os.path.exists(src):
                shutil.copy2(
                    src,
                    f"{export_dir}/results/{name}_{filename}"
                )

        figures_dir = f"{run_dir}/figures"
        if os.path.isdir(figures_dir):
            for filename in os.listdir(figures_dir):
                shutil.copy2(
                    f"{figures_dir}/{filename}",
                    f"{export_dir}/figures/{filename}"
                )

    print("\nGitHub export ready:", export_dir)
    print("Completed runs:", len(df), "/ 11")


In [ ]:
import os
import pandas as pd

target_path = f"{base_dir}/{best['Model']}/submission.csv"

if os.path.exists(target_path):
    print("✅ Found submission.csv in your Google Drive!")
    sub_df = pd.read_csv(target_path)
    print(f"Total Rows: {len(sub_df)}")
    print("\n--- First 10 Rows of the Submission ---")
    display(sub_df.head(10))
    print("\n--- Sentiment Class Distribution ---")
    print(sub_df['predictions'].value_counts())
else:
    print(f"❌ The submission file does not exist yet at: {target_path}")
    print("Please run the BERT training cell (chkBsFMm67Br) first to train the model, make predictions on the test set, and automatically save the file to your Google Drive!")